In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 - Silver: Tratamento e Padronização
# MAGIC
# MAGIC Objetivo:
# MAGIC - Ler as tabelas Delta da camada Bronze
# MAGIC - Padronizar chaves de relacionamento
# MAGIC - Tratar datas, valores numéricos, status e campos categóricos
# MAGIC - Normalizar estruturas aninhadas de JSON
# MAGIC - Gravar tabelas Silver confiáveis para modelagem Gold

from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
BASE_PATH = "/Volumes/case/default/case_data"

BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)

display(dbutils.fs.ls(BRONZE_PATH))

BRONZE_PATH: /Volumes/case/default/case_data/bronze
SILVER_PATH: /Volumes/case/default/case_data/silver


path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/bronze/canais/,canais/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/clientes/,clientes/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/entregas/,entregas/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/ocorrencias/,ocorrencias/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/pedidos_cabecalho/,pedidos_cabecalho/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/pedidos_itens/,pedidos_itens/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/produtos/,produtos/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/regioes/,regioes/,0,1781141841312
dbfs:/Volumes/case/default/case_data/bronze/vendedores/,vendedores/,0,1781141841312


In [0]:
bronze_pedidos_cabecalho = spark.read.format("delta").load(f"{BRONZE_PATH}/pedidos_cabecalho")
bronze_pedidos_itens = spark.read.format("delta").load(f"{BRONZE_PATH}/pedidos_itens")
bronze_vendedores = spark.read.format("delta").load(f"{BRONZE_PATH}/vendedores")
bronze_regioes = spark.read.format("delta").load(f"{BRONZE_PATH}/regioes")
bronze_produtos = spark.read.format("delta").load(f"{BRONZE_PATH}/produtos")
bronze_entregas = spark.read.format("delta").load(f"{BRONZE_PATH}/entregas")
bronze_ocorrencias = spark.read.format("delta").load(f"{BRONZE_PATH}/ocorrencias")
bronze_clientes = spark.read.format("delta").load(f"{BRONZE_PATH}/clientes")
bronze_canais = spark.read.format("delta").load(f"{BRONZE_PATH}/canais")

print("Tabelas Bronze carregadas com sucesso.")

Tabelas Bronze carregadas com sucesso.


# MAGIC %md
# MAGIC ## Funções auxiliares de padronização
# MAGIC
# MAGIC As funções abaixo centralizam regras reutilizáveis de limpeza e normalização:
# MAGIC - padronização de IDs
# MAGIC - tratamento de nulos textuais
# MAGIC - conversão segura de valores numéricos
# MAGIC - parsing de datas em múltiplos formatos
# MAGIC - normalização de status e estados

In [0]:
def normalizar_id(coluna):
    return F.upper(F.trim(F.col(coluna).cast("string")))

def tratar_string_nula(coluna):
    c = F.trim(F.col(coluna).cast("string"))
    return (
        F.when(F.lower(c).isin("nan", "none", "null", "", "nat"), None)
         .otherwise(c)
    )

def tratar_valor_decimal(coluna):
    c = F.trim(F.col(coluna).cast("string"))
    c_lower = F.lower(c)
    c_decimal = F.regexp_replace(c, ",", ".")

    return (
        F.when(
            c_lower.isin(
                "unknown",
                "nan",
                "none",
                "null",
                "",
                "nat",
                "n/a",
                "na",
                "não informado",
                "nao informado",
                "-"
            ),
            None
        )
        .when(
            c_decimal.rlike(r"^-?\d+(\.\d+)?$"),
            c_decimal.cast("double")
        )
        .otherwise(None)
    )

def parse_data_multiplo_formato(coluna):
    c = F.col(coluna).cast("string")
    return F.coalesce(
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss")),
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(c, F.lit("dd/MM/yyyy HH:mm")),
        F.try_to_timestamp(c, F.lit("dd/MM/yyyy")),
        F.try_to_timestamp(c, F.lit("yyyy/MM/dd"))
    )

def normalizar_status(coluna):
    c = F.lower(F.trim(F.col(coluna).cast("string")))
    return (
        F.when(c.isin("ativo", "sim", "s", "yes", "true", "1"), "ATIVO")
         .when(c.isin("inativo", "nao", "não", "n", "no", "false", "0"), "INATIVO")
         .when(c.isin("descontinuado"), "DESCONTINUADO")
         .when(c.isin("faturado"), "FATURADO")
         .when(c.isin("em_separacao", "em separacao", "em separação"), "EM_SEPARACAO")
         .when(c.isin("cancelado", "cancelled"), "CANCELADO")
         .when(c.isin("delivered", "entregue"), "ENTREGUE")
         .when(c.isin("in_transit", "em_transito", "em trânsito"), "EM_TRANSITO")
         .when(c.isin("atrasado"), "ATRASADO")
         .when(c.isin("closed", "fechado"), "FECHADO")
         .when(c.isin("open", "aberto"), "ABERTO")
         .otherwise(F.upper(F.trim(F.col(coluna).cast("string"))))
    )

def normalizar_estado(coluna):
    c = F.lower(F.trim(F.col(coluna).cast("string")))
    c = F.translate(c, "áàãâéêíóôõúç", "aaaaeeiooouc")
    
    return (
        F.when(c.isin("sp", "sao paulo"), "SP")
         .when(c.isin("rj", "rio de janeiro"), "RJ")
         .when(c.isin("mg", "minas gerais"), "MG")
         .when(c.isin("pr", "parana"), "PR")
         .when(c.isin("sc", "santa catarina", "s. catarina", "sta catarina"), "SC")
         .when(c.isin("am", "amazonas"), "AM")
         .when(c.isin("ba", "bahia"), "BA")
         .when(c.isin("go", "goias"), "GO")
         .otherwise(F.upper(F.trim(F.col(coluna).cast("string"))))
    )

In [0]:
silver_pedidos_cabecalho = (
    bronze_pedidos_cabecalho
    .select(
        normalizar_id("order_id").alias("order_id"),
        normalizar_id("customer_code").alias("customer_id"),
        normalizar_id("seller_id").alias("seller_id"),
        parse_data_multiplo_formato("order_date").alias("order_date"),
        parse_data_multiplo_formato("promised_date").alias("promised_date"),
        normalizar_status("status_order").alias("status_order"),
        tratar_valor_decimal("gross_amount").alias("gross_amount"),
        tratar_valor_decimal("discount_amount").alias("discount_amount"),
        tratar_valor_decimal("net_amount").alias("net_amount"),
        F.get_json_object(F.col("payment_details"), "$.priority").alias("payment_priority"),
        F.get_json_object(F.col("payment_details"), "$.source").alias("payment_source"),
        parse_data_multiplo_formato("last_update").alias("last_update"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
    .withColumn("dias_ate_promessa", F.datediff(F.col("promised_date"), F.col("order_date")))
    .withColumn(
        "fl_cancelado",
        F.when(F.col("status_order") == "CANCELADO", F.lit(1)).otherwise(F.lit(0))
    )
)

display(silver_pedidos_cabecalho.limit(5))

order_id,customer_id,seller_id,order_date,promised_date,status_order,gross_amount,discount_amount,net_amount,payment_priority,payment_source,last_update,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,dias_ate_promessa,fl_cancelado
O00001,C0002,V036,2025-07-31T00:00:00.000Z,2025-08-06T00:00:00.000Z,FATURADO,10788.64,0.0,10788.64,null,null,2025-08-20T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:50:56.590Z,6,0
O00002,C0023,V008,2025-04-13T00:00:00.000Z,2025-04-26T00:00:00.000Z,EM_SEPARACAO,14462.32,0.0,14462.32,null,null,2025-04-26T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:50:56.590Z,13,0
O00003,C0127,V012,2025-04-01T00:00:00.000Z,2025-04-16T00:00:00.000Z,FATURADO,13802.56,1994.35,11808.21,null,null,2025-04-19T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:50:56.590Z,15,0
O00004,C0165,V035,2025-11-05T00:00:00.000Z,2025-11-07T00:00:00.000Z,CANCELADO,13933.98,670.75,13263.23,null,null,2025-11-11T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:50:56.590Z,2,1
O00005,C0048,V001,2026-02-19T00:00:00.000Z,2026-02-21T00:00:00.000Z,null,5027.19,0.0,5027.19,null,null,2026-02-25T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:50:56.590Z,2,0


In [0]:
silver_pedidos_itens = (
    bronze_pedidos_itens
    .select(
        normalizar_id("order_id").alias("order_id"),
        F.col("item_seq").cast("int").alias("item_seq"),
        normalizar_id("product_code").alias("product_id"),
        F.col("quantity").cast("double").alias("quantity"),
        tratar_valor_decimal("unit_price").alias("unit_price"),
        tratar_valor_decimal("total_item").alias("total_item"),
        normalizar_status("item_status").alias("item_status"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
    .withColumn(
        "valor_calculado_item",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
    .withColumn(
        "dif_total_item",
        F.round(F.col("total_item") - F.col("valor_calculado_item"), 2)
    )
)

display(silver_pedidos_itens.limit(5))

order_id,item_seq,product_id,quantity,unit_price,total_item,item_status,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,valor_calculado_item,dif_total_item
O00001,1,P0027,5.0,453.12,2265.6,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:53:01.152Z,2265.6,0.0
O00001,2,P0035,10.0,2278.1,22781.0,null,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:53:01.152Z,22781.0,0.0
O00001,3,P0063,2.0,629.8,1259.6,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:53:01.152Z,1259.6,0.0
O00001,4,P0001,3.0,1502.54,4507.62,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:53:01.152Z,4507.62,0.0
O00002,1,P0032,3.0,1274.78,3824.34,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:53:01.152Z,3824.34,0.0


In [0]:
silver_produtos = (
    bronze_produtos
    .select(
        normalizar_id("product.product_id").alias("product_id"),
        tratar_string_nula("product.name").alias("product_name"),
        tratar_string_nula("product.category").alias("category"),
        tratar_string_nula("product.subcategory").alias("subcategory"),
        normalizar_status("product.status").alias("product_status"),
        tratar_valor_decimal("pricing.list_price").alias("list_price"),
        tratar_string_nula("pricing.currency").alias("currency"),
        tratar_string_nula("attributes.family").alias("product_family"),
        F.col("attributes.tags").alias("tags"),
        F.concat_ws(",", F.col("attributes.tags")).alias("tags_text"),
        parse_data_multiplo_formato("updated_at").alias("updated_at"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
)

display(silver_produtos.limit(5))

product_id,product_name,category,subcategory,product_status,list_price,currency,product_family,tags,tags_text,updated_at,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento
P0001,Produto 1,Software,CRM,ATIVO,376.26,BRL,Core,"List(b2b, legacy, cloud)","b2b,legacy,cloud",2026-03-13T00:00:00.000Z,2026-06-11T01:31:56.438Z,cadastro_produtos_api,cadastro_produtos_api_dump.json,2026-06-11T01:53:16.600Z
P0002,Produto 2,Hardware,Sensor,ATIVO,2934.56,BRL,Premium,"List(b2b, cloud)","b2b,cloud",2025-03-08T00:00:00.000Z,2026-06-11T01:31:56.438Z,cadastro_produtos_api,cadastro_produtos_api_dump.json,2026-06-11T01:53:16.600Z
P0003,Produto 3,Software,ETL,null,1914.56,BRL,Legacy,List(legacy),legacy,2025-02-17T00:00:00.000Z,2026-06-11T01:31:56.438Z,cadastro_produtos_api,cadastro_produtos_api_dump.json,2026-06-11T01:53:16.600Z
P0004,Produto 4,Assinatura,Anual,ATIVO,1844.86,BRL,Premium,List(b2b),b2b,2026-01-28T00:00:00.000Z,2026-06-11T01:31:56.438Z,cadastro_produtos_api,cadastro_produtos_api_dump.json,2026-06-11T01:53:16.600Z
P0005,Produto 5,Assinatura,Mensal,INATIVO,4999.88,BRL,Premium,List(),,2025-06-01T00:00:00.000Z,2026-06-11T01:31:56.438Z,cadastro_produtos_api,cadastro_produtos_api_dump.json,2026-06-11T01:53:16.600Z


In [0]:
silver_clientes = (
    bronze_clientes
    .select(
        normalizar_id("customer_id").alias("customer_id"),
        tratar_string_nula("nome_cliente").alias("nome_cliente"),
        tratar_string_nula("segmento").alias("segmento"),
        F.upper(tratar_string_nula("porte")).alias("porte"),
        tratar_string_nula("cidade").alias("cidade"),
        normalizar_estado("estado").alias("estado"),
        parse_data_multiplo_formato("data_cadastro").alias("data_cadastro"),
        tratar_string_nula("email").alias("email"),
        normalizar_status("status_cliente").alias("status_cliente"),
        parse_data_multiplo_formato("updated_at").alias("updated_at"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
)

display(silver_clientes.limit(5))

customer_id,nome_cliente,segmento,porte,cidade,estado,data_cadastro,email,status_cliente,updated_at,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento
C0001,Cliente 1,null,GRANDE,Florianópolis,SC,2024-01-26T00:00:00.000Z,cliente1@empresa.com,NAN,2025-02-22T00:00:00.000Z,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,2026-06-11T01:56:12.426Z
C0002,Cliente 2,Educação,GRANDE,Curitiba,PR,2024-03-30T00:00:00.000Z,cliente2@empresa.com,ATIVO,2025-04-30T00:00:00.000Z,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,2026-06-11T01:56:12.426Z
C0003,Cliente 3,Varejo,null,Curitiba,PR,2025-09-08T00:00:00.000Z,cliente3@empresa.com,NAN,2025-10-07T00:00:00.000Z,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,2026-06-11T01:56:12.426Z
C0004,Cliente 4,Saúde,null,Uberlândia,MG,2024-08-13T00:00:00.000Z,cliente4@empresa.com,NAN,2025-08-05T00:00:00.000Z,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,2026-06-11T01:56:12.426Z
C0005,Cliente 5,Indústria,MÉDIA,Niterói,RJ,2024-10-11T00:00:00.000Z,cliente5@empresa.com,INATIVO,2025-02-19T00:00:00.000Z,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,2026-06-11T01:56:12.426Z


In [0]:
silver_canais = (
    bronze_canais
    .select(
        normalizar_id("id_canal").alias("channel_id"),
        tratar_string_nula("nome_canal").alias("nome_canal"),
        tratar_string_nula("tipo_canal").alias("tipo_canal"),
        normalizar_status("ativo").alias("status_canal"),
        tratar_string_nula("observacao").alias("observacao"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
)

display(silver_canais.limit(5))

channel_id,nome_canal,tipo_canal,status_canal,observacao,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento
CH01,Inside Sales,Direto,ATIVO,null,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,2026-06-11T01:56:21.137Z
CH02,Parceiros,Indireto,ATIVO,null,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,2026-06-11T01:56:21.137Z
CH03,Marketplace,Digital,ATIVO,null,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,2026-06-11T01:56:21.137Z
CH04,Field Sales,Direto,INATIVO,canal legado,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,2026-06-11T01:56:21.137Z
CH05,E-commerce,digital,ATIVO,null,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,2026-06-11T01:56:21.137Z


In [0]:
silver_vendedores = (
    bronze_vendedores
    .select(
        normalizar_id("seller_id").alias("seller_id"),
        tratar_string_nula("seller_name").alias("seller_name"),
        normalizar_id("canal_id").alias("channel_id"),
        normalizar_id("regional_code").alias("regional_code"),
        parse_data_multiplo_formato("hire_date").alias("hire_date"),
        normalizar_status("status").alias("status_vendedor"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
)

display(silver_vendedores.limit(5))

seller_id,seller_name,channel_id,regional_code,hire_date,status_vendedor,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento
V001,Vendedor 1,CH01,SUL,2024-06-27T00:00:00.000Z,null,2026-06-11T01:31:50.469Z,vendedores,vendedores.csv,2026-06-11T01:56:32.690Z
V002,Vendedor 2,CH04,S,2023-12-16T00:00:00.000Z,ATIVO,2026-06-11T01:31:50.469Z,vendedores,vendedores.csv,2026-06-11T01:56:32.690Z
V003,Vendedor 3,CH03,SUL,2023-11-29T00:00:00.000Z,ATIVO,2026-06-11T01:31:50.469Z,vendedores,vendedores.csv,2026-06-11T01:56:32.690Z
V004,Vendedor 4,CH02,NE,2025-02-11T00:00:00.000Z,ATIVO,2026-06-11T01:31:50.469Z,vendedores,vendedores.csv,2026-06-11T01:56:32.690Z
V005,Vendedor 5,CH03,S,2024-11-08T00:00:00.000Z,null,2026-06-11T01:31:50.469Z,vendedores,vendedores.csv,2026-06-11T01:56:32.690Z


In [0]:
silver_regioes = (
    bronze_regioes
    .select(
        normalizar_id("regional_code").alias("regional_code"),
        tratar_string_nula("regional_name").alias("regional_name"),
        normalizar_estado("state").alias("state"),
        tratar_string_nula("manager_name").alias("manager_name"),
        normalizar_status("active_flag").alias("status_regiao"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
)

display(silver_regioes.limit(10))

regional_code,regional_name,state,manager_name,status_regiao,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento
N,Norte,AM,Mariana Lopes,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z
NE,Nordeste,BA,Carlos Lima,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z
SE,Sudeste,SP,Ana Costa,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z
S,Sul,SC,Rafael Souza,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z
CO,Centro-Oeste,GO,Paulo Teixeira,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z
SUL,Região Sul,SC,Rafael Souza,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z
SE,Sudeste,SP,Ana Costa,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z
XX,null,null,Sem gestor,INATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:56:40.375Z


In [0]:
silver_entregas = (
    bronze_entregas
    .select(
        normalizar_id("delivery_id").alias("delivery_id"),
        normalizar_id("order_ref").alias("order_id"),
        tratar_string_nula("carrier.name").alias("carrier_name"),
        tratar_string_nula("carrier.mode").alias("delivery_mode"),
        normalizar_status("delivery_status").alias("delivery_status"),
        parse_data_multiplo_formato("timestamps.shipped_at").alias("shipped_at"),
        parse_data_multiplo_formato("timestamps.delivered_at").alias("delivered_at"),
        normalizar_estado("destination.state").alias("destination_state"),
        tratar_string_nula("destination.city").alias("destination_city"),
        tratar_valor_decimal("cost").alias("delivery_cost"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
    .withColumn(
        "delivery_days",
        F.datediff(F.col("delivered_at"), F.col("shipped_at"))
    )
    .withColumn(
        "fl_entregue",
        F.when(F.col("delivered_at").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "fl_entrega_atrasada",
        F.when(F.col("delivery_status") == "ATRASADO", F.lit(1)).otherwise(F.lit(0))
    )
)

display(silver_entregas.limit(5))

delivery_id,order_id,carrier_name,delivery_mode,delivery_status,shipped_at,delivered_at,destination_state,destination_city,delivery_cost,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,delivery_days,fl_entregue,fl_entrega_atrasada
D00001,O00129,null,Rodoviário,null,2026-01-21T00:00:00.000Z,2026-01-21T00:00:00.000Z,RJ,Rio de Janeiro,56.54,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:57:06.491Z,0,1,0
D00002,O00213,TransSul,null,ENTREGUE,2025-02-18T00:00:00.000Z,2025-02-27T00:00:00.000Z,SC,Blumenau,46.34,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:57:06.491Z,9,1,0
D00003,O00048,LogFast,Rodoviário,ATRASADO,2025-03-23T00:00:00.000Z,2025-04-01T00:00:00.000Z,PR,Maringá,248.24,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:57:06.491Z,9,1,1
D00004,O00250,null,Aéreo,null,2025-02-23T00:00:00.000Z,2025-03-05T00:00:00.000Z,SC,Florianópolis,346.48,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:57:06.491Z,10,1,0
D00005,O00172,EntregaJá,Aéreo,ENTREGUE,2025-01-03T00:00:00.000Z,2025-01-13T00:00:00.000Z,PR,Curitiba,218.51,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:57:06.491Z,10,1,0


In [0]:
silver_ocorrencias = (
    bronze_ocorrencias
    .select(
        normalizar_id("ticket_id").alias("ticket_id"),
        normalizar_id("order_id").alias("order_id"),
        tratar_string_nula("event_type").alias("event_type"),
        parse_data_multiplo_formato("created_at").alias("created_at"),
        F.upper(tratar_string_nula("severity")).alias("severity"),
        normalizar_status("status").alias("status_ocorrencia"),
        F.col("_data_ingestao"),
        F.col("_nome_fonte"),
        F.col("_arquivo_origem"),
        F.current_timestamp().alias("_data_tratamento")
    )
    .withColumn(
        "fl_ocorrencia_aberta",
        F.when(F.col("status_ocorrencia") == "ABERTO", F.lit(1)).otherwise(F.lit(0))
    )
)

display(silver_ocorrencias.limit(5))

ticket_id,order_id,event_type,created_at,severity,status_ocorrencia,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,fl_ocorrencia_aberta
T00001,O00385,refund,2025-01-04T05:00:00.000Z,HIGH,FECHADO,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:58:25.408Z,0
T00002,O00051,troca,2026-02-12T00:00:00.000Z,MEDIUM,ABERTO,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:58:25.408Z,1
T00003,O00181,refund,2026-02-07T00:00:00.000Z,null,null,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:58:25.408Z,0
T00004,O00371,delay,2025-12-18T00:00:00.000Z,LOW,null,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:58:25.408Z,0
T00005,O00055,null,2025-02-27T16:00:00.000Z,MEDIUM,ABERTO,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:58:25.408Z,1


In [0]:
# Gravação das tabelas tratadas na camada Silver.
# A Silver contém dados padronizados, tipados e prontos para a modelagem analítica Gold.

tabelas_silver = {
    "pedidos_cabecalho": silver_pedidos_cabecalho,
    "pedidos_itens": silver_pedidos_itens,
    "produtos": silver_produtos,
    "clientes": silver_clientes,
    "canais": silver_canais,
    "vendedores": silver_vendedores,
    "regioes": silver_regioes,
    "entregas": silver_entregas,
    "ocorrencias": silver_ocorrencias
}

for nome_tabela, df in tabelas_silver.items():
    caminho_delta = f"{SILVER_PATH}/{nome_tabela}"
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(caminho_delta)
    )
    
    print(
        f"Tabela Silver gravada: {nome_tabela} | "
        f"Linhas: {df.count()} | "
        f"Caminho: {caminho_delta}"
    )

Tabela Silver gravada: pedidos_cabecalho | Linhas: 403 | Caminho: /Volumes/case/default/case_data/silver/pedidos_cabecalho
Tabela Silver gravada: pedidos_itens | Linhas: 995 | Caminho: /Volumes/case/default/case_data/silver/pedidos_itens
Tabela Silver gravada: produtos | Linhas: 72 | Caminho: /Volumes/case/default/case_data/silver/produtos
Tabela Silver gravada: clientes | Linhas: 183 | Caminho: /Volumes/case/default/case_data/silver/clientes
Tabela Silver gravada: canais | Linhas: 8 | Caminho: /Volumes/case/default/case_data/silver/canais
Tabela Silver gravada: vendedores | Linhas: 42 | Caminho: /Volumes/case/default/case_data/silver/vendedores
Tabela Silver gravada: regioes | Linhas: 8 | Caminho: /Volumes/case/default/case_data/silver/regioes
Tabela Silver gravada: entregas | Linhas: 322 | Caminho: /Volumes/case/default/case_data/silver/entregas
Tabela Silver gravada: ocorrencias | Linhas: 270 | Caminho: /Volumes/case/default/case_data/silver/ocorrencias


In [0]:
display(dbutils.fs.ls(SILVER_PATH))

for nome_tabela in tabelas_silver.keys():
    df_validacao = spark.read.format("delta").load(f"{SILVER_PATH}/{nome_tabela}")
    
    print("=" * 100)
    print(f"silver.{nome_tabela}")
    print("Linhas:", df_validacao.count())
    df_validacao.printSchema()
    display(df_validacao.limit(5))

path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/silver/canais/,canais/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/clientes/,clientes/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/entregas/,entregas/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/ocorrencias/,ocorrencias/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/pedidos_cabecalho/,pedidos_cabecalho/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/pedidos_itens/,pedidos_itens/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/produtos/,produtos/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/regioes/,regioes/,0,1781143174811
dbfs:/Volumes/case/default/case_data/silver/vendedores/,vendedores/,0,1781143174811


silver.pedidos_cabecalho
Linhas: 403
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- promised_date: timestamp (nullable = true)
 |-- status_order: string (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- payment_priority: string (nullable = true)
 |-- payment_source: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_tratamento: timestamp (nullable = true)
 |-- dias_ate_promessa: integer (nullable = true)
 |-- fl_cancelado: integer (nullable = true)



order_id,customer_id,seller_id,order_date,promised_date,status_order,gross_amount,discount_amount,net_amount,payment_priority,payment_source,last_update,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,dias_ate_promessa,fl_cancelado
O00001,C0002,V036,2025-07-31T00:00:00.000Z,2025-08-06T00:00:00.000Z,FATURADO,10788.64,0.0,10788.64,null,null,2025-08-20T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:58:39.691Z,6,0
O00002,C0023,V008,2025-04-13T00:00:00.000Z,2025-04-26T00:00:00.000Z,EM_SEPARACAO,14462.32,0.0,14462.32,null,null,2025-04-26T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:58:39.691Z,13,0
O00003,C0127,V012,2025-04-01T00:00:00.000Z,2025-04-16T00:00:00.000Z,FATURADO,13802.56,1994.35,11808.21,null,null,2025-04-19T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:58:39.691Z,15,0
O00004,C0165,V035,2025-11-05T00:00:00.000Z,2025-11-07T00:00:00.000Z,CANCELADO,13933.98,670.75,13263.23,null,null,2025-11-11T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:58:39.691Z,2,1
O00005,C0048,V001,2026-02-19T00:00:00.000Z,2026-02-21T00:00:00.000Z,null,5027.19,0.0,5027.19,null,null,2026-02-25T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,2026-06-11T01:58:39.691Z,2,0


silver.pedidos_itens
Linhas: 995
root
 |-- order_id: string (nullable = true)
 |-- item_seq: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_item: double (nullable = true)
 |-- item_status: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_tratamento: timestamp (nullable = true)
 |-- valor_calculado_item: double (nullable = true)
 |-- dif_total_item: double (nullable = true)



order_id,item_seq,product_id,quantity,unit_price,total_item,item_status,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,valor_calculado_item,dif_total_item
O00001,1,P0027,5.0,453.12,2265.6,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:58:42.746Z,2265.6,0.0
O00001,2,P0035,10.0,2278.1,22781.0,null,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:58:42.746Z,22781.0,0.0
O00001,3,P0063,2.0,629.8,1259.6,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:58:42.746Z,1259.6,0.0
O00001,4,P0001,3.0,1502.54,4507.62,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:58:42.746Z,4507.62,0.0
O00002,1,P0032,3.0,1274.78,3824.34,ATIVO,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,2026-06-11T01:58:42.746Z,3824.34,0.0


...

regional_code,regional_name,state,manager_name,status_regiao,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento
N,Norte,AM,Mariana Lopes,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:58:55.537Z
NE,Nordeste,BA,Carlos Lima,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:58:55.537Z
SE,Sudeste,SP,Ana Costa,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:58:55.537Z
S,Sul,SC,Rafael Souza,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:58:55.537Z
CO,Centro-Oeste,GO,Paulo Teixeira,ATIVO,2026-06-11T01:31:53.576Z,legado_regioes,legado_regioes_pipe.txt,2026-06-11T01:58:55.537Z


silver.entregas
Linhas: 322
root
 |-- delivery_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- delivery_mode: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- shipped_at: timestamp (nullable = true)
 |-- delivered_at: timestamp (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- delivery_cost: double (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_tratamento: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- fl_entregue: integer (nullable = true)
 |-- fl_entrega_atrasada: integer (nullable = true)



delivery_id,order_id,carrier_name,delivery_mode,delivery_status,shipped_at,delivered_at,destination_state,destination_city,delivery_cost,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,delivery_days,fl_entregue,fl_entrega_atrasada
D00001,O00129,null,Rodoviário,null,2026-01-21T00:00:00.000Z,2026-01-21T00:00:00.000Z,RJ,Rio de Janeiro,56.54,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:58:57.944Z,0,1,0
D00002,O00213,TransSul,null,ENTREGUE,2025-02-18T00:00:00.000Z,2025-02-27T00:00:00.000Z,SC,Blumenau,46.34,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:58:57.944Z,9,1,0
D00003,O00048,LogFast,Rodoviário,ATRASADO,2025-03-23T00:00:00.000Z,2025-04-01T00:00:00.000Z,PR,Maringá,248.24,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:58:57.944Z,9,1,1
D00004,O00250,null,Aéreo,null,2025-02-23T00:00:00.000Z,2025-03-05T00:00:00.000Z,SC,Florianópolis,346.48,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:58:57.944Z,10,1,0
D00005,O00172,EntregaJá,Aéreo,ENTREGUE,2025-01-03T00:00:00.000Z,2025-01-13T00:00:00.000Z,PR,Curitiba,218.51,2026-06-11T01:31:59.092Z,logistica_entregas,logistica_entregas.json,2026-06-11T01:58:57.944Z,10,1,0


silver.ocorrencias
Linhas: 270
root
 |-- ticket_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- severity: string (nullable = true)
 |-- status_ocorrencia: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_tratamento: timestamp (nullable = true)
 |-- fl_ocorrencia_aberta: integer (nullable = true)



ticket_id,order_id,event_type,created_at,severity,status_ocorrencia,_data_ingestao,_nome_fonte,_arquivo_origem,_data_tratamento,fl_ocorrencia_aberta
T00001,O00385,refund,2025-01-04T05:00:00.000Z,HIGH,FECHADO,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:59:00.412Z,0
T00002,O00051,troca,2026-02-12T00:00:00.000Z,MEDIUM,ABERTO,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:59:00.412Z,1
T00003,O00181,refund,2026-02-07T00:00:00.000Z,null,null,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:59:00.412Z,0
T00004,O00371,delay,2025-12-18T00:00:00.000Z,LOW,null,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:59:00.412Z,0
T00005,O00055,null,2025-02-27T16:00:00.000Z,MEDIUM,ABERTO,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,2026-06-11T01:59:00.412Z,1
